# mixedbench demo

The registry, one object and one problem, end to end:
browse the registry, load the table of `lme4/sleepstudy`,
then build, materialise and load the problem `lme4/sleepstudy/1`.
No R is needed; the one download is lme4 from CRAN.

In [ ]:
%pip install -q mixedbench

## Browse the registry

The package ships the registry files; `SUMMARY.md` opens with the corpus in one paragraph.

In [1]:
from pathlib import Path

import pandas as pd
from mixedbench import registry

paragraphs = registry.path('SUMMARY.md').read_text().split('\n\n')
[corpus] = [paragraph for paragraph in paragraphs if paragraph.startswith('**Corpus.**')]
print(corpus)

**Corpus.** 714 registry rows over 641 datasets and 941 grouping
factors; per tier R 252, T 244, D 104, E 101, O 13. Excluded: 2133 objects (no grouping structure 1164, derived or duplicated table 622, unreadable format 192, not the paper's data 46, simulated data 44, response missing or undocumented 36, time series without replication 28, fewer than two groups 1).


A registry row is one documented model on one table, named `<source>/<object>/<model_index>`.

In [2]:
rows = registry.rows()
columns = ['id', 'tier', 'response_type', 'formula']
records = [{column: getattr(row, column) for column in columns} for row in rows]
pd.DataFrame(records).head()

,id,tier,response_type,formula
0,lme4/Arabidopsis/1,R,count,total.fruits ~ nutrient*amd + rack + status + ...
1,lme4/Dyestuff/1,R,continuous,Yield ~ 1 + (1|Batch)
2,lme4/Pastes/1,R,continuous,strength ~ (1|batch/cask)
3,lme4/Penicillin/1,R,continuous,diameter ~ (1|plate) + (1|sample)
4,lme4/VerbAgg/1,R,binary,r2 ~ (Anger + Gender + btype + situ)^2 + (1|id...


Every source is fetched at a pin: a CRAN version, a git revision or a file's sha256.
The sources of Tier R, the CRAN packages:

In [3]:
sources = registry.sources()
labels = sorted({row.source for row in rows if row.tier == 'R'})
pinned = [
    {'source': label, 'kind': sources[label].kind, 'pin': sources[label].pin} for label in labels
]
print(f'{len(labels)} sources')
pd.DataFrame(pinned).head()

22 sources


,source,kind,pin
0,HLMdiag,cran,0.5.1
1,HSAUR3,cran,1.0-16
2,MASS,cran,7.3-66
3,MEMSS,cran,0.9-4
4,SASmixed,cran,1.0-5


## Load one object

The loader fetches only the object's file into `cache/`, checks it against its sha256
and returns the table with the row's declared column types.

In [4]:
cache = Path('cache')
[row] = [row for row in rows if row.id == 'lme4/sleepstudy/1']
print(row.formula)
table = registry.Tables(cache).load(row)
table.head()

Reaction ~ Days + (Days|Subject)


,Reaction,Days,Subject
0,249.5600,0.0,308
1,258.7047,1.0,308
2,250.8006,2.0,308
3,321.4398,3.0,308
4,356.8519,4.0,308


## Build, materialise and load one problem

A problem is the table a model is fitted to and what it takes to fit and score it.
Materialised once, it loads from its two files alone: no loader, no network.

In [5]:
from mixedbench.contract import Problem, load, materialise

materialise(Problem.build(row, table), cache)  # the contract hash

'8111b74ef393568a2cb445ef9f10cc4f538abd8d6a6d48c6b91181fac5a8f330'

In [6]:
problem = load('lme4/sleepstudy/1', cache)
print('design: ', problem.design)
print('response:', problem.response, f'({problem.family}, {problem.link} link)')
print('factors: ', problem.factors)
problem.table.head()

design:  ('Intercept', 'Days')
response: Reaction (gaussian, identity link)
factors:  ('Subject',)


,Intercept,Days,Reaction,Subject,rows,groups
0,1.0,0.0,249.5600,0,seen,seen
1,1.0,1.0,258.7047,0,seen,seen
2,1.0,2.0,250.8006,0,seen,seen
3,1.0,3.0,321.4398,0,seen,seen
4,1.0,4.0,356.8519,0,seen,seen


One prior serves every problem, per brms class, on the standardised scale:

In [7]:
problem.prior

{'Intercept': Prior(distribution='normal', parameters=(0, 2.5)),
 'b': Prior(distribution='normal', parameters=(0, 2.5)),
 'cor': Prior(distribution='lkj', parameters=(1,)),
 'sd': Prior(distribution='student_t', parameters=(3, 0, 2.5)),
 'sigma': Prior(distribution='student_t', parameters=(3, 0, 2.5))}

Each regime standardises over the rows a fit sees:
the full table, or the seen rows of the held-out-rows and held-out-groups splits.

In [8]:
scales = [
    {'regime': regime, 'column': column, 'mean': scale.mean, 'sd': scale.sd}
    for regime, columns in problem.scales.items()
    for column, scale in columns.items()
]
pd.DataFrame(scales)

,regime,column,mean,sd
0,full,Days,4.500000,2.880293
1,full,Reaction,298.507892,56.328757
2,groups,Days,4.500000,2.881299
3,groups,Reaction,298.563104,58.626440
4,rows,Days,4.425926,2.838960
5,rows,Reaction,298.159565,55.611442


## What 1.1 adds

- **Reference posteriors:** brms draws for every problem of suite 1.1 that passes the diagnostics bar.
- **Metrics:** posterior and predictive scores against the references, with the chain-split null as their floor.
- **Opponent lines:** the scores of standard methods over the suite, to read a new method against.
- **This notebook** gains a toy fit of `lme4/sleepstudy/1` scored against its reference draws.